# Weekly SQL monitoring: champion and three challengers

This executable example fits a synthetic Tweedie burn-cost model, saves its baseline in SQL, and removes the original model files. A later SQL source snapshot then supplies one champion score and three saved challenger packages. The generated `monitoring.py` script and notebook `07_model_monitoring.ipynb` repeat the same run without duplicating publications.

The model has grouped regions, ordered bonus-malus levels with an explicit Unknown level, and a continuous age spline. Burn cost is loss per exposure; exposure is the fitting weight.

This notebook uses an isolated SQLite database. A guarded demo helper simulates the initial published/current deployment. Challenger packages remain `LOCAL_AUDIT`, as ordinary local notebook APIs require. No SQL Server connection or production deployment is tested.

| Variant | Role | Coefficients | Smoothing lambdas | Data-driven knots |
|---|---|---|---|---|
| STATIC_SCORE | Champion observation | Baseline | Baseline | Baseline |
| FROZEN_REFIT | Saved challenger | Refit | Fixed | Fixed |
| REESTIMATE_LAMBDA | Saved challenger | Refit | Re-estimated | Fixed |
| FULL_ADAPTIVE | Saved challenger | Refit | Re-estimated | Repositioned |

Run all cells in the project's Python environment. Output starts under `state/sql_monitoring_demo`; every rerun uses a new directory. The example creates its own model project and leaves existing user notebooks untouched.


In [1]:
import json
import os
import sqlite3
import subprocess
import sys
from pathlib import Path

import pandas as pd
from sqlalchemy import text
from superglm import Categorical, OrderedCategorical, Spline, SuperGLM, Tweedie, collapse_levels

from pricing_pipeline import notebook as pricing_api
from pricing_pipeline.modeling.monitoring import check_monitoring_data
from pricing_pipeline.scaffold.config import ScaffoldOptions
from pricing_pipeline.scaffold.service import scaffold_pricing_model

PROJECT_ROOT = next(
    directory for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / "pyproject.toml").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT))
from scripts.demo_sql_monitoring import (
    create_demo_directory,
    export_sql_tables,
    simulate_demo_deployment,
    synthetic_burn_cost,
)

if "display" not in globals():
    try:
        from IPython.display import display
    except ImportError:
        display = print

OUTPUT = create_demo_directory(PROJECT_ROOT / "state" / "sql_monitoring_demo")
DEMO_PROJECT = OUTPUT / "project"
DEMO_PROJECT.mkdir()
(DEMO_PROJECT / "pyproject.toml").write_text('[project]\nname="sql-monitoring-demo"\nversion="0.0.0"\n')
options = ScaffoldOptions(
    model_name="SQL_DEMO_BURN_COST", model_label="Synthetic burn cost",
    target_name="burn_cost", model_type="burn_cost", deployment_slot="DEMO_CURRENT",
    database_mode="local", root=DEMO_PROJECT,
)
scaffold_pricing_model(options)
MODEL_DIR = DEMO_PROJECT / "pricing_models" / "sql_demo_burn_cost"

# Emulate an existing model folder that predates the monitoring additions.
# These two files were just generated inside this fresh demo directory.
(MODEL_DIR / "monitoring.py").unlink()
(MODEL_DIR / "07_model_monitoring.ipynb").unlink()
original_notebooks = {path: path.read_bytes() for path in MODEL_DIR.glob("*.ipynb")}
pricing = pricing_api.connect(mode="local", local_root=MODEL_DIR / ".local")
print(f"All demo output: {OUTPUT}")


All demo output: /home/max/projects/superglm-pricing-pipeline/.worktrees/sql-monitoring-baseline/state/sql_monitoring_demo/run-px0zy1n_


## 1. Fit and save the initial champion candidate

The baseline contains 480 synthetic policies dated 2026-08-31. A compound Poisson-gamma generator supplies non-negative losses, including zeros. Both fitting and export use exposure weights. The model uses REML, two validation folds, and `retain_fit_state=False`.


In [2]:
baseline_df = synthetic_burn_cost(rows=480, seed=1701, as_of="2026-08-31")
dataset = pricing_api.PricingDataset(
    baseline_df, name="synthetic_burn_cost", source="synthetic_demo",
    key="policy_id", as_of="as_of",
)

feature_names = ("region", "bonus_malus", "driver_age")
spec = pricing_api.PricingModelSpec(
    name="SQL_DEMO_BURN_COST",
    label="Synthetic burn cost",
    model_type="burn_cost",
    target="burn_cost",
    deployment_slot="DEMO_CURRENT",
    dataset=dataset,
    features=feature_names,
    sample_weight_column="exposure",
    export_weight_column="exposure",
    fit_mode="fit_reml",
    validation=pricing_api.ValidationSplitConfig.kfold(n_splits=2, random_state=19),
)

glm = SuperGLM(
    family=Tweedie(p=1.5),
    selection_penalty=0.0,
    retain_fit_state=False,
    features={
        "region": Categorical(
            base="North",
            grouping=collapse_levels(baseline_df.region, groups={"EastWest": ["East", "West"]}),
        ),
        "bonus_malus": OrderedCategorical(
            order=["0", "1", "2", "3", "4"],
            specials=["Unknown"],
            base="0",
            basis=Spline("cr", k=3, knot_strategy="quantile"),
        ),
        "driver_age": Spline("cr", k=3, knot_strategy="quantile"),
    },
)
registered = pricing_api.register_model(
    pricing, spec, source_root=MODEL_DIR,
    created_by="synthetic-demo",
)
candidate = pricing_api.fit_model(
    pricing, model=registered, frame=baseline_df,
    superglm_model=glm, created_by="synthetic-demo",
)
display(baseline_df.head(5))
display(pd.DataFrame([candidate.metrics]).round(4))


,policy_id,as_of,region,bonus_malus,driver_age,exposure,burn_cost
0,0,2026-08-31,South,0,21.383962,1.276383,1350.072511
1,1,2026-08-31,South,3,74.903695,0.806185,1464.469776
2,2,2026-08-31,East,0,26.887111,0.695058,0.000000
3,3,2026-08-31,North,4,42.363722,1.155516,0.000000
4,4,2026-08-31,West,0,24.524095,1.271834,407.928154


,cv_mean_deviance,cv_mean_nll,cv_mean_gini,cv_pooled_deviance,cv_pooled_nll,cv_std_deviance,cv_std_nll,cv_std_gini,cv_oof_coverage,fit_converged,fit_n_iter,fit_deviance,fit_effective_df,fit_phi,fit_log_likelihood,fit_null_log_likelihood,fit_null_deviance,fit_explained_deviance,fit_pearson_chi2,fit_n_obs,fit_likelihood_size,fit_reml_enabled,fit_reml_converged,fit_reml_n_iter
0,42.1141,5.3492,0.094,42.1141,5.3492,1.0361,0.0127,0.0427,1.0,1.0,1.0,19178.1841,6.7312,31.9715,-2550.5706,-2559.208,19730.4851,0.028,15131.1096,480.0,480.0,1.0,1.0,9.0


Saving allocates the package, successful model run, and recipe revision. The same transaction captures immutable monitoring JSON with scoring state, refit settings, aggregate reference profiles, and verified source lineage. No training rows or serialized model object are stored in that snapshot.


In [3]:
saved = pricing_api.save_model_version(pricing, candidate)
with pricing.engine.connect() as connection:
    publication_rows = pd.read_sql_query(text("""
        SELECT mr.model_run_id, mr.model_version, mr.run_status,
               rp.rate_package_id, rp.package_status, recipe.recipe_revision,
               baseline.capture_status, baseline.snapshot_schema_version,
               LENGTH(baseline.snapshot_json) AS snapshot_characters,
               baseline.snapshot_sha256
        FROM pricing.MODEL_RUN AS mr
        JOIN pricing.PRICING_RATE_PACKAGE AS rp ON rp.rate_package_id=mr.rate_package_id
        JOIN pricing.MODEL_RECIPE AS recipe ON recipe.recipe_id=mr.recipe_id
        JOIN pricing.MODEL_MONITORING_BASELINE AS baseline ON baseline.model_run_id=mr.model_run_id
    """), connection)
display(publication_rows)
assert publication_rows.capture_status.tolist() == ["CAPTURED"]
assert saved.package_status == "LOCAL_AUDIT"


,model_run_id,model_version,run_status,rate_package_id,package_status,recipe_revision,capture_status,snapshot_schema_version,snapshot_characters,snapshot_sha256
0,1,v1,SUCCESS,1,LOCAL_AUDIT,1,CAPTURED,2,32270,5bf0436aaf1dfc54a8949bd43db87a31622f0047fa90ce...


## 2. Add monitoring to the existing model folder

Calling the scaffolder again adds `monitoring.py` and notebook 07. The existing notebooks and initial publication artifacts keep their exact bytes. Configure the generated loader to read a separate SQLite source table in stable policy order and use its recorded snapshot date.

In a real project, replace this source query with the current data query and enrichment steps. The generated module owns the connection settings and loader; both the scheduler and notebook 07 call its `run()` function.


In [4]:
publication_files = [
    Path(candidate.completed_build.candidate_artifact_path),
    Path(candidate.completed_build.rating_workbook_path),
    Path(candidate.completed_build.publication_receipt_path),
]
original_artifacts = {path: path.read_bytes() for path in publication_files}
upgrade = scaffold_pricing_model(options)
assert {path.name for path in upgrade.created_files} == {"monitoring.py", "07_model_monitoring.ipynb"}
assert all(path.read_bytes() == original for path, original in original_notebooks.items())
assert all(path.read_bytes() == original for path, original in original_artifacts.items())

monitoring_df = synthetic_burn_cost(rows=360, seed=1702, as_of="2026-09-15", monitoring=True)
with sqlite3.connect(DEMO_PROJECT / "current_source.sqlite") as source_connection:
    monitoring_df.to_sql("current_source", source_connection, index=False, if_exists="fail")

script = MODEL_DIR / "monitoring.py"
script_source = script.read_text()
loader_start = script_source.index("def load_dataset()")
loader_end = script_source.index("\ndef run():", loader_start)
loader = '''def load_dataset() -> PricingDataset:
    """Read this demo's current source in stable key order with its source date."""
    import sqlite3
    import pandas as pd

    with sqlite3.connect(PROJECT_ROOT / "current_source.sqlite") as connection:
        df = pd.read_sql_query(
            "SELECT policy_id, as_of, region, bonus_malus, driver_age, exposure, burn_cost "
            "FROM current_source ORDER BY policy_id",
            connection,
        )
    return PricingDataset(
        df=df, name="synthetic_burn_cost", source="synthetic_demo",
        key="policy_id", as_of="as_of",
    )

'''
script.write_text(script_source[:loader_start] + loader + script_source[loader_end:])
print("Added:", [path.name for path in upgrade.created_files])
print("Original notebooks and publication artifacts preserved.")
print(script.read_text()[loader_start:script.read_text().index("\ndef run():")])


Added: ['07_model_monitoring.ipynb', 'monitoring.py']
Original notebooks and publication artifacts preserved.
def load_dataset() -> PricingDataset:
    """Read this demo's current source in stable key order with its source date."""
    import sqlite3
    import pandas as pd

    with sqlite3.connect(PROJECT_ROOT / "current_source.sqlite") as connection:
        df = pd.read_sql_query(
            "SELECT policy_id, as_of, region, bonus_malus, driver_age, exposure, burn_cost "
            "FROM current_source ORDER BY policy_id",
            connection,
        )
    return PricingDataset(
        df=df, name="synthetic_burn_cost", source="synthetic_demo",
        key="policy_id", as_of="as_of",
    )




## 3. Simulate the initial deployment and remove its files

The helper checks the SQLite dialect, the marked demo directory, and every attached database path before creating the one simulated deployment. It leaves database guards enabled. Ordinary local APIs cannot deploy this model.

Only the three original publication files are removed. The model object, candidate, and old training dataframe are also discarded. Recurring monitoring below has SQL plus the later source data.


In [5]:
deployment_id = simulate_demo_deployment(
    pricing, saved, directory=OUTPUT, slot=registered.config.deployment_slot,
)
for path in publication_files:
    assert path.resolve().is_relative_to(OUTPUT)
    path.unlink()
del candidate, glm, baseline_df, dataset, spec, registered, original_artifacts
print(f"Simulated deployment: {deployment_id}")
assert not any(path.exists() for path in publication_files)
print("Original publication files remaining: 0")


Simulated deployment: 1
Original publication files remaining: 0


## 4. Read the later source and check the SQL baseline

The new snapshot has 360 policies dated 2026-09-15. Its generator applies 12% claims inflation and a larger North region share; actual outcomes remain random. The date comes from the source column, not the day the script runs.

The saved snapshot currently requires the exact SuperGLM version and a matching Python major/minor version. Runtime checks remain enabled even though original model files are no longer required.


In [6]:
sys.path.insert(0, str(DEMO_PROJECT))
from pricing_models.sql_demo_burn_cost import monitoring

fresh = monitoring.load_dataset()
model = pricing_api.load_registered_model(
    pricing, model_name="SQL_DEMO_BURN_COST", model_label="Synthetic burn cost",
    deployment_slot="DEMO_CURRENT", source_root=MODEL_DIR,
)
baseline = pricing_api.load_monitoring_baseline(pricing, model=model)
preflight = check_monitoring_data(
    baseline, fresh.df.loc[:, list(baseline.feature_names)], sample_weight=fresh.df.exposure,
)
display(pd.DataFrame([{
    "baseline_model_run_id": baseline.model_run_id,
    "baseline_deployment_id": baseline.deployment_id,
    "snapshot_sha256": baseline.snapshot_sha256,
    "source_date": fresh.df.as_of.iloc[0],
    "rows": len(fresh.df),
}]))
display(preflight.issues)
display(preflight.drift.round(4))
preflight.raise_for_errors()


,baseline_model_run_id,baseline_deployment_id,snapshot_sha256,source_date,rows
0,1,1,5bf0436aaf1dfc54a8949bd43db87a31622f0047fa90ce...,2026-09-15,360


,feature,severity,code,message,affected_rows,affected_weight
0,driver_age,warning,SPLINE_RANGE_LOSS,Spline feature 'driver_age' now has positive-w...,0,None
1,region,warning,CATEGORICAL_DRIFT,Feature 'region' has a large categorical mix c...,0,None


,feature,row_distance,weight_distance,threshold,needs_review
0,region,0.216,0.2140,0.2,True
1,bonus_malus,0.000,0.0117,0.2,False


## 5. Save the weekly champion observation and three challengers

`run_monitoring` checks every variant and finishes all four fits before writing evidence. It then saves four observations and publishes the exact three refitted estimators as challenger packages. Each challenger gets its own model run, package version and SQL baseline snapshot. All inherit the original model definition revision. No additional fit or validation split changes those estimators during export.

The static observation keeps the champion's baseline identity and has no candidate package. Each challenger row contains both its new package/model identifiers and the original `baseline_*` identifiers. Publication and observation writes use individual transactions, so a later failure can leave earlier results saved. Exact retries reuse saved observations and packages.

The lower-level `run_monitoring_fit` and `persist_monitoring_fit` APIs remain evidence-only. This notebook uses the scheduled wrapper that also publishes challengers. A remote run creates published challengers; this local demonstration creates `LOCAL_AUDIT` packages. Neither automatically replaces the champion.


In [7]:
report = pricing_api.run_monitoring(pricing, model=model, dataset=fresh)
display(report.runs)
display(report.metrics.pivot(index="variant", columns="metric_name", values="metric_value").round(4))
assert report.runs.role.value_counts().to_dict() == {"CHALLENGER": 3, "CHAMPION": 1}
challengers = report.runs.query("role == 'CHALLENGER'")
assert challengers.model_run_id.notna().all()
assert challengers.package_status.eq("LOCAL_AUDIT").all()
assert report.runs.query("role == 'CHAMPION'").rate_package_id.isna().all()
initial_observation_ids = set(report.runs.monitor_run_id)
initial_challenger_ids = set(challengers.model_run_id.astype(str))

assert report.runs.definition_revision.eq(1).all()


,variant,role,model_name,definition_revision,fit_version,model_id,baseline_model_run_id,baseline_deployment_id,baseline_rate_package_id,baseline_package_version,deployment_slot,monitor_run_id,fit_contract_id,run_signature_sha256,deduplicated,model_run_id,rate_package_id,package_version,package_status,publication_reused
0,STATIC_SCORE,CHAMPION,SQL_DEMO_BURN_COST,1,None,1,1,1,1,1,DEMO_CURRENT,b5c85e34-2554-42c9-b515-4cf9a554344a,2f34354d-7533-4000-828b-b2144a8c9b30,694c633e92bd64345831a5eb68f0cb7416e24b11fe250a...,False,None,None,None,None,None
1,FROZEN_REFIT,CHALLENGER,SQL_DEMO_BURN_COST,1,v2,1,1,1,1,1,DEMO_CURRENT,294652b9-7aeb-41c1-aa4a-06f11f223f1b,2f34354d-7533-4000-828b-b2144a8c9b30,873863e77b481e9043cc6fc331f7362a7df68fa999876c...,False,2,2,2,LOCAL_AUDIT,False
2,REESTIMATE_LAMBDA,CHALLENGER,SQL_DEMO_BURN_COST,1,v3,1,1,1,1,1,DEMO_CURRENT,c7e5411a-f502-4847-8974-0fd07b87c8be,2f34354d-7533-4000-828b-b2144a8c9b30,ad49ed9a189abb6f9271aa8c64b0384ff23fc513cab8e7...,False,3,3,3,LOCAL_AUDIT,False
3,FULL_ADAPTIVE,CHALLENGER,SQL_DEMO_BURN_COST,1,v4,1,1,1,1,1,DEMO_CURRENT,ccb59082-f87f-45ef-89c2-19a40000fbb7,2f34354d-7533-4000-828b-b2144a8c9b30,7dc62cab5be7574dc76afafb7cd145d58553fbd1214384...,False,4,4,4,LOCAL_AUDIT,False


metric_name,deviance,explained_deviance,log_likelihood,null_deviance,row_count,sample_weight_sum,sample_weighted_mean_observed,sample_weighted_mean_prediction,sample_weighted_sum_observed,sample_weighted_sum_prediction
variant,,,,,,,,,,
FROZEN_REFIT,15125.4820,0.0307,-1991.2410,15604.6636,360.0,351.6873,419.6702,418.0809,147592.6582,147033.7439
FULL_ADAPTIVE,15110.5814,0.0317,-1991.0774,15604.6636,360.0,351.6873,419.6702,418.3801,147592.6582,147138.9575
REESTIMATE_LAMBDA,15109.3857,0.0317,-1991.0616,15604.6636,360.0,351.6873,419.6702,418.3757,147592.6582,147137.4266
STATIC_SCORE,15407.1833,0.0127,-1997.1128,15604.6636,360.0,351.6873,419.6702,376.3774,147592.6582,132367.1309


STATIC_SCORE CATEGORICAL_DRIFT: Feature 'region' has a large categorical mix change. Check upstream SQL definitions and portfolio composition; a new baseline may be needed. Drift alone does not identify the cause.
FROZEN_REFIT SPLINE_RANGE_LOSS: Spline feature 'driver_age' now has positive-weight range [20.0537, 77.9803] against saved domain [18.082, 79.917]. Parts of the saved curve have no new observations supporting them; review before interpreting changes there.
FROZEN_REFIT CATEGORICAL_DRIFT: Feature 'region' has a large categorical mix change. Check upstream SQL definitions and portfolio composition; a new baseline may be needed. Drift alone does not identify the cause.
REESTIMATE_LAMBDA SPLINE_RANGE_LOSS: Spline feature 'driver_age' now has positive-weight range [20.0537, 77.9803] against saved domain [18.082, 79.917]. Parts of the saved curve have no new observations supporting them; review before interpreting changes there.
REESTIMATE_LAMBDA CATEGORICAL_DRIFT: Feature 'region'

## 6. Execute the generated script from an unrelated directory

This is the actual generated `monitoring.py`, invoked with the current environment's absolute Python interpreter. For this source-checkout demo, `PYTHONPATH` points to the package source. An installed project only needs its normal environment.

The script reads the same dated source table and writes a separate scheduler log. The assertions below confirm that this retry creates no extra model runs, packages, publication links, or observations.


In [8]:
unrelated_cwd = OUTPUT / "unrelated_scheduler_directory"
unrelated_cwd.mkdir()
environment = os.environ.copy()
environment["PYTHONPATH"] = str(PROJECT_ROOT / "src")
process = subprocess.run(
    [sys.executable, str(script.resolve())],
    cwd=unrelated_cwd, env=environment, capture_output=True, text=True, check=False,
)
(OUTPUT / "scheduler_stdout.txt").write_text(process.stdout)
(OUTPUT / "scheduler_stderr.txt").write_text(process.stderr)
print("Command:", [sys.executable, str(script.resolve())])
print("Working directory:", unrelated_cwd)
print("Exit code:", process.returncode)
if process.returncode:
    print(process.stdout, process.stderr)
process.check_returncode()
with pricing.engine.connect() as connection:
    scheduled_ids = {str(value) for value in connection.execute(text("SELECT model_run_id FROM pricing.MODEL_MONITOR_PUBLICATION")).scalars()}
    assert scheduled_ids == initial_challenger_ids
    assert connection.execute(text("SELECT COUNT(*) FROM pricing.MODEL_RUN")).scalar_one() == 4
    assert connection.execute(text("SELECT COUNT(*) FROM pricing.PRICING_RATE_PACKAGE")).scalar_one() == 4
    assert connection.execute(text("SELECT COUNT(*) FROM pricing.MODEL_MONITOR_RUN")).scalar_one() == 4
logs = sorted((MODEL_DIR / ".local" / "monitoring_logs").glob("*.log"))
assert logs
print("Scheduler log:", logs[-1])
print("Script retry preserved all three challenger package identities.")


Command: ['/home/max/projects/superglm-pricing-pipeline/.worktrees/model-recipes/.venv/bin/python', '/home/max/projects/superglm-pricing-pipeline/.worktrees/sql-monitoring-baseline/state/sql_monitoring_demo/run-px0zy1n_/project/pricing_models/sql_demo_burn_cost/monitoring.py']
Working directory: /home/max/projects/superglm-pricing-pipeline/.worktrees/sql-monitoring-baseline/state/sql_monitoring_demo/run-px0zy1n_/unrelated_scheduler_directory
Exit code: 0
Scheduler log: /home/max/projects/superglm-pricing-pipeline/.worktrees/sql-monitoring-baseline/state/sql_monitoring_demo/run-px0zy1n_/project/pricing_models/sql_demo_burn_cost/.local/monitoring_logs/monitoring_20260915T203515_294891Z_e7785fdc.log
Script retry preserved all three challenger package identities.


## 7. Execute notebook 07 against the same run

The following cell executes the unmodified code cells from the generated notebook. Its `monitoring.run()` call uses the same loader and workflow as the scheduler. All four observations and all three challenger publications must be reused. Open the generated notebook separately to use it interactively.


In [9]:
notebook07_path = MODEL_DIR / "07_model_monitoring.ipynb"
notebook07 = json.loads(notebook07_path.read_text())
namespace07 = {"__name__": "__main__", "display": display}
previous_cwd = Path.cwd()
executed_cells = 0
try:
    os.chdir(DEMO_PROJECT)
    for index, cell in enumerate(notebook07["cells"]):
        if cell["cell_type"] == "code":
            exec(compile("".join(cell["source"]), f"{notebook07_path}:cell-{index}", "exec"), namespace07)  # noqa: S102 - locally generated notebook
            executed_cells += 1
finally:
    os.chdir(previous_cwd)
retry_report = namespace07["report"]
assert retry_report.manifest_id == report.manifest_id
assert set(retry_report.runs.monitor_run_id) == initial_observation_ids
assert retry_report.runs.deduplicated.all()
assert retry_report.runs.query("role == 'CHALLENGER'").publication_reused.all()
assert set(retry_report.runs.query("role == 'CHALLENGER'").model_run_id.astype(str)) == initial_challenger_ids
assert all(path.read_bytes() == original for path, original in original_notebooks.items())
print(f"Executed {executed_cells} generated notebook code cells; all results reused.")


{'manifest_id': 'synthetic_burn_cost_20260915_84e8c95118'}

,variant,role,model_name,definition_revision,fit_version,model_id,baseline_model_run_id,baseline_deployment_id,baseline_rate_package_id,baseline_package_version,deployment_slot,monitor_run_id,fit_contract_id,run_signature_sha256,deduplicated,model_run_id,rate_package_id,package_version,package_status,publication_reused
0,STATIC_SCORE,CHAMPION,SQL_DEMO_BURN_COST,1,None,1,1,1,1,1,DEMO_CURRENT,b5c85e34-2554-42c9-b515-4cf9a554344a,2f34354d-7533-4000-828b-b2144a8c9b30,694c633e92bd64345831a5eb68f0cb7416e24b11fe250a...,True,None,None,None,None,None
1,FROZEN_REFIT,CHALLENGER,SQL_DEMO_BURN_COST,1,v2,1,1,1,1,1,DEMO_CURRENT,294652b9-7aeb-41c1-aa4a-06f11f223f1b,2f34354d-7533-4000-828b-b2144a8c9b30,873863e77b481e9043cc6fc331f7362a7df68fa999876c...,True,2,2,2,LOCAL_AUDIT,True
2,REESTIMATE_LAMBDA,CHALLENGER,SQL_DEMO_BURN_COST,1,v3,1,1,1,1,1,DEMO_CURRENT,c7e5411a-f502-4847-8974-0fd07b87c8be,2f34354d-7533-4000-828b-b2144a8c9b30,ad49ed9a189abb6f9271aa8c64b0384ff23fc513cab8e7...,True,3,3,3,LOCAL_AUDIT,True
3,FULL_ADAPTIVE,CHALLENGER,SQL_DEMO_BURN_COST,1,v4,1,1,1,1,1,DEMO_CURRENT,ccb59082-f87f-45ef-89c2-19a40000fbb7,2f34354d-7533-4000-828b-b2144a8c9b30,7dc62cab5be7574dc76afafb7cd145d58553fbd1214384...,True,4,4,4,LOCAL_AUDIT,True


,variant,metric_name,metric_value
0,STATIC_SCORE,deviance,15407.183348
1,STATIC_SCORE,explained_deviance,0.012655
2,STATIC_SCORE,log_likelihood,-1997.112822
3,STATIC_SCORE,null_deviance,15604.663575
4,STATIC_SCORE,row_count,360.000000
5,STATIC_SCORE,sample_weight_sum,351.687285
6,STATIC_SCORE,sample_weighted_mean_observed,419.670157
7,STATIC_SCORE,sample_weighted_mean_prediction,376.377357
8,STATIC_SCORE,sample_weighted_sum_observed,147592.658212
9,STATIC_SCORE,sample_weighted_sum_prediction,132367.130912


,feature,severity,code,message,affected_rows,affected_weight,variant
0,region,warning,CATEGORICAL_DRIFT,Feature 'region' has a large categorical mix c...,0,None,STATIC_SCORE
1,driver_age,warning,SPLINE_RANGE_LOSS,Spline feature 'driver_age' now has positive-w...,0,None,FROZEN_REFIT
2,region,warning,CATEGORICAL_DRIFT,Feature 'region' has a large categorical mix c...,0,None,FROZEN_REFIT
3,driver_age,warning,SPLINE_RANGE_LOSS,Spline feature 'driver_age' now has positive-w...,0,None,REESTIMATE_LAMBDA
4,region,warning,CATEGORICAL_DRIFT,Feature 'region' has a large categorical mix c...,0,None,REESTIMATE_LAMBDA
5,driver_age,warning,SPLINE_RANGE_LOSS,Spline feature 'driver_age' now has positive-w...,0,None,FULL_ADAPTIVE
6,region,warning,CATEGORICAL_DRIFT,Feature 'region' has a large categorical mix c...,0,None,FULL_ADAPTIVE


,feature,row_distance,weight_distance,threshold,needs_review,variant
0,region,0.215972,0.214025,0.2,True,STATIC_SCORE
1,bonus_malus,0.000000,0.011660,0.2,False,STATIC_SCORE
2,region,0.215972,0.214025,0.2,True,FROZEN_REFIT
3,bonus_malus,0.000000,0.011660,0.2,False,FROZEN_REFIT
4,region,0.215972,0.214025,0.2,True,REESTIMATE_LAMBDA
5,bonus_malus,0.000000,0.011660,0.2,False,REESTIMATE_LAMBDA
6,region,0.215972,0.214025,0.2,True,FULL_ADAPTIVE
7,bonus_malus,0.000000,0.011660,0.2,False,FULL_ADAPTIVE


Configuration: /home/max/projects/superglm-pricing-pipeline/.worktrees/sql-monitoring-baseline/state/sql_monitoring_demo/run-px0zy1n_/project/pricing_models/sql_demo_burn_cost/monitoring.py
Current champion and published challengers
Metrics
Warnings and data checks
Categorical drift
/home/max/projects/superglm-pricing-pipeline/.worktrees/model-recipes/.venv/bin/python /home/max/projects/superglm-pricing-pipeline/.worktrees/sql-monitoring-baseline/state/sql_monitoring_demo/run-px0zy1n_/project/pricing_models/sql_demo_burn_cost/monitoring.py
Python executable: /home/max/projects/superglm-pricing-pipeline/.worktrees/model-recipes/.venv/bin/python
File argument: "/home/max/projects/superglm-pricing-pipeline/.worktrees/sql-monitoring-baseline/state/sql_monitoring_demo/run-px0zy1n_/project/pricing_models/sql_demo_burn_cost/monitoring.py"
Logs: /home/max/projects/superglm-pricing-pipeline/.worktrees/sql-monitoring-baseline/state/sql_monitoring_demo/run-px0zy1n_/project/pricing_models/sql_demo

STATIC_SCORE CATEGORICAL_DRIFT: Feature 'region' has a large categorical mix change. Check upstream SQL definitions and portfolio composition; a new baseline may be needed. Drift alone does not identify the cause.
FROZEN_REFIT SPLINE_RANGE_LOSS: Spline feature 'driver_age' now has positive-weight range [20.0537, 77.9803] against saved domain [18.082, 79.917]. Parts of the saved curve have no new observations supporting them; review before interpreting changes there.
FROZEN_REFIT CATEGORICAL_DRIFT: Feature 'region' has a large categorical mix change. Check upstream SQL definitions and portfolio composition; a new baseline may be needed. Drift alone does not identify the cause.
REESTIMATE_LAMBDA SPLINE_RANGE_LOSS: Spline feature 'driver_age' now has positive-weight range [20.0537, 77.9803] against saved domain [18.082, 79.917]. Parts of the saved curve have no new observations supporting them; review before interpreting changes there.
REESTIMATE_LAMBDA CATEGORICAL_DRIFT: Feature 'region'

## 8. Inspect the SQL evidence and challenger view

Each saved observation has comparable relativities on the baseline grid. Lower deviance here describes fit to this monitoring snapshot and does not establish out-of-sample improvement. The challenger view connects each dated candidate to its original champion and the current deployment.


In [10]:
with pricing.engine.connect() as connection:
    metric_rows = pd.read_sql_query(text("""
        SELECT run.variant_code AS variant, metric.metric_name, metric.metric_value
        FROM pricing.MODEL_MONITOR_RUN AS run
        JOIN pricing.MODEL_MONITOR_METRIC AS metric ON metric.monitor_run_id=run.monitor_run_id
    """), connection)
    relativity_rows = pd.read_sql_query(text("""
        SELECT run.variant_code AS variant, rel.term_name, rel.point_label,
               rel.point_numeric, rel.relativity
        FROM pricing.MODEL_MONITOR_RUN AS run
        JOIN pricing.MODEL_MONITOR_RELATIVITY AS rel ON rel.monitor_run_id=run.monitor_run_id
        ORDER BY rel.term_name, rel.point_label, rel.point_numeric, run.variant_code
    """), connection)
metrics = metric_rows.pivot(index="variant", columns="metric_name", values="metric_value")
display(metrics[[
    "row_count", "sample_weighted_mean_observed",
    "sample_weighted_mean_prediction", "deviance", "explained_deviance",
]].round(4))
region_comparison = relativity_rows.query("term_name == 'region'").pivot(
    index="point_label", columns="variant", values="relativity",
)
display(region_comparison.round(4))

with pricing.engine.connect() as connection:
    challenger_view = pd.read_sql_query(text("SELECT * FROM pricing.V_MODEL_CHALLENGER ORDER BY model_run_id"), connection)
display(challenger_view)
assert len(challenger_view) == 3
assert not challenger_view.is_current_champion.any()

with pricing.engine.connect() as connection:
    registry = pd.read_sql_query(text("SELECT * FROM pricing.V_MODEL_REGISTRY ORDER BY package_version"), connection)
display(registry[["model_name", "role", "definition_revision", "refit_type", "data_as_of_date", "published_at", "package_version"]])
assert len(registry) == 4
assert registry.definition_revision.eq(1).all()
assert registry.role.tolist() == ["CHAMPION", "CHALLENGER", "CHALLENGER", "CHALLENGER"]


metric_name,row_count,sample_weighted_mean_observed,sample_weighted_mean_prediction,deviance,explained_deviance
variant,,,,,
FROZEN_REFIT,360.0,419.6702,418.0809,15125.4820,0.0307
FULL_ADAPTIVE,360.0,419.6702,418.3801,15110.5814,0.0317
REESTIMATE_LAMBDA,360.0,419.6702,418.3757,15109.3857,0.0317
STATIC_SCORE,360.0,419.6702,376.3774,15407.1833,0.0127


variant,FROZEN_REFIT,FULL_ADAPTIVE,REESTIMATE_LAMBDA,STATIC_SCORE
point_label,,,,
East,0.8805,0.8848,0.8850,0.7097
North,1.0000,1.0000,1.0000,1.0000
South,1.1526,1.1553,1.1553,0.8030
West,0.8805,0.8848,0.8850,0.7097


,monitor_run_id,model_run_id,model_id,model_name,model_version,rate_package_id,package_version,package_status,variant_code,data_as_of_date,baseline_data_as_of_date,baseline_model_run_id,baseline_rate_package_id,baseline_deployment_id,deployment_slot,current_deployment_id,current_rate_package_id,is_current_champion,created_ts,created_by
0,294652b9-7aeb-41c1-aa4a-06f11f223f1b,2,1,SQL_DEMO_BURN_COST,v2,2,2,LOCAL_AUDIT,FROZEN_REFIT,2026-09-15,2026-08-31,1,1,1,DEMO_CURRENT,1,1,0,2026-09-15 20:35:12,max
1,c7e5411a-f502-4847-8974-0fd07b87c8be,3,1,SQL_DEMO_BURN_COST,v3,3,3,LOCAL_AUDIT,REESTIMATE_LAMBDA,2026-09-15,2026-08-31,1,1,1,DEMO_CURRENT,1,1,0,2026-09-15 20:35:12,max
2,ccb59082-f87f-45ef-89c2-19a40000fbb7,4,1,SQL_DEMO_BURN_COST,v4,4,4,LOCAL_AUDIT,FULL_ADAPTIVE,2026-09-15,2026-08-31,1,1,1,DEMO_CURRENT,1,1,0,2026-09-15 20:35:13,max


,model_name,role,definition_revision,refit_type,data_as_of_date,published_at,package_version
0,SQL_DEMO_BURN_COST,CHAMPION,1,Analyst fit,2026-08-31,2026-09-15 20:35:11,1
1,SQL_DEMO_BURN_COST,CHALLENGER,1,Coefficients only,2026-09-15,2026-09-15 20:35:12,2
2,SQL_DEMO_BURN_COST,CHALLENGER,1,Coefficients and smoothing,2026-09-15,2026-09-15 20:35:12,3
3,SQL_DEMO_BURN_COST,CHALLENGER,1,Full refit,2026-09-15,2026-09-15 20:35:13,4


## 9. Export the actual SQL tables

The Excel workbook includes a table guide, a readable feature sheet for every saved snapshot, and up to 20 actual rows from each SQL table or view. The guide reports total counts, exported counts, sort order, purpose, and key links. Complete snapshots are saved separately by model-run ID. Long Excel cells have labelled previews with JSON sidecar paths.

`MODEL_MONITOR_PUBLICATION` links each refit observation to its saved challenger. `V_MODEL_CHALLENGER` joins candidate, baseline, dataset date, and current champion identities. SQL Server places observation, publication-link, and fit-contract tables in `mlops`; the SQLite mirror places them in `pricing`.


In [11]:
workbook_path, table_guide = export_sql_tables(pricing, directory=OUTPUT, limit=20)
display(table_guide[["SQL Server table", "total rows", "rows exported", "extract scope"]])
with pricing.engine.connect() as connection:
    counts = {
        table: connection.execute(text(f"SELECT COUNT(*) FROM pricing.{table}")).scalar_one()
        for table in ["MODEL_RUN", "PRICING_RATE_PACKAGE", "MODEL_MONITORING_BASELINE", "MODEL_MONITOR_RUN", "MODEL_MONITOR_PUBLICATION"]
    }
    current_package = connection.execute(text(
        "SELECT rate_package_id FROM pricing.PRICING_MODEL_DEPLOYMENT WHERE effective_to_ts IS NULL"
    )).scalar_one()
assert counts == {
    "MODEL_RUN": 4, "PRICING_RATE_PACKAGE": 4, "MODEL_MONITORING_BASELINE": 4,
    "MODEL_MONITOR_RUN": 4, "MODEL_MONITOR_PUBLICATION": 3,
}
assert current_package == saved.rate_package_id
assert not any(path.exists() for path in publication_files)
evidence = {
    "output_directory": str(OUTPUT),
    "workbook": str(workbook_path),
    "script": str(script),
    "script_exit_code": process.returncode,
    "script_cwd": str(unrelated_cwd),
    "scheduler_log": str(logs[-1]),
    "generated_notebook": str(notebook07_path),
    "generated_notebook_code_cells_executed": executed_cells,
    "all_observations_reused": bool(retry_report.runs.deduplicated.all()),
    "all_challenger_publications_reused": bool(retry_report.runs.query("role == 'CHALLENGER'").publication_reused.all()),
    "sql_counts": counts,
    "current_package_unchanged": True,
    "original_model_files_remaining": 0,
}
(OUTPUT / "execution_evidence.json").write_text(json.dumps(evidence, indent=2) + "\n")
print(f"Excel workbook: {workbook_path}")
print(f"Full snapshots: {OUTPUT / 'baseline_snapshot_model_run_*.json'}")
print(f"SQLite database: {MODEL_DIR / '.local' / 'pricing.sqlite'}")
print(f"Execution evidence: {OUTPUT / 'execution_evidence.json'}")
pricing.engine.dispose()


,SQL Server table,total rows,rows exported,extract scope
0,pricing.V_MODEL_REGISTRY,4,4,All rows
1,pricing.PRICING_MODEL,1,1,All rows
2,pricing.MODEL_RECIPE,1,1,All rows
3,pricing.MODEL_RUN,4,4,All rows
4,pricing.PRICING_RATE_PACKAGE,4,4,All rows
5,pricing.MODEL_MONITORING_BASELINE,4,4,All rows
6,pricing.PRICING_MODEL_DEPLOYMENT,1,1,All rows
7,pricing.DATASET_MANIFEST,2,2,All rows
8,mlops.MODEL_FIT_CONTRACT,1,1,All rows
9,mlops.MODEL_MONITOR_RUN,4,4,All rows


Excel workbook: /home/max/projects/superglm-pricing-pipeline/.worktrees/sql-monitoring-baseline/state/sql_monitoring_demo/run-px0zy1n_/sql_tables.xlsx
Full snapshots: /home/max/projects/superglm-pricing-pipeline/.worktrees/sql-monitoring-baseline/state/sql_monitoring_demo/run-px0zy1n_/baseline_snapshot_model_run_*.json
SQLite database: /home/max/projects/superglm-pricing-pipeline/.worktrees/sql-monitoring-baseline/state/sql_monitoring_demo/run-px0zy1n_/project/pricing_models/sql_demo_burn_cost/.local/pricing.sqlite
Execution evidence: /home/max/projects/superglm-pricing-pipeline/.worktrees/sql-monitoring-baseline/state/sql_monitoring_demo/run-px0zy1n_/execution_evidence.json


For an existing remote champion, ordinary publication and deployment already supply its SQL lineage. Configure that model's generated `monitoring.py` and schedule it with the project's Python interpreter. Each new dated dataset saves a static champion observation and three published challenger packages. Review and promotion remain separate actions. Older published models without a SQL snapshot can use `capture_existing_monitoring_baseline` once with a verified workbench candidate, without refitting.
